In [1]:
import pyspark.sql.functions as F

from pyspark.sql import SparkSession
import os
from pyspark.sql.types import *
from pyspark.sql.functions import col, year, month, when

In [2]:
spark = SparkSession \
    .builder \
    .config("spark.streaming.stopGracefullyOnShutdown", True) \
    .config("spark.sql.shuffle.partitions", 4) \
    .master("local[*]") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/11 23:00:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
#Data Schema from one of the files
static_df = (
    spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .option("ignoreTrailingWhiteSpace", "true")
        .option("ignoreLeadingWhiteSpace", "true")
        .option("mode", "DROPMALFORMED")
        .load("Data/report_2014_1.csv")
)

schema = static_df.schema

In [4]:


# Path to the directory containing the CSV files
input_path = "Data"
#Files processed per trigger
NUM_FILES_PER_TRIGGER = 1
# Read the streaming DataFrame from the directory
streaming_df = spark.readStream \
    .option("maxFilesPerTrigger", NUM_FILES_PER_TRIGGER) \
    .option("header", "true") \
    .format("csv") \
    .schema(schema) \
    .option("ignoreTrailingWhiteSpace", "true") \
    .option("ignoreLeadingWhiteSpace", "true") \
    .option("mode", "DROPMALFORMED") \
    .load(input_path)


In [5]:
column_mapping = {
    "FL_DATE": ("FlightDate", DateType()),
    "OP_CARRIER": ("Reporting_Airline", StringType()),
    "OP_CARRIER_FL_NUM": ("Flight_Number_Reporting_Airline", IntegerType()),
    "ORIGIN_CITY": ("OriginCityName", StringType()),
    "ORIGIN": ("Origin", StringType()),
    "DEST_CITY": ("DestCityName", StringType()),
    "DEST": ("Dest", StringType()),
    "CRS_DEP_TIME": ("CRSDepTime", IntegerType()),
    "DEP_TIME": ("DepTime", FloatType()),
    "DEP_DELAY": ("DepDelay", FloatType()),
    "TAXI_OUT": ("TaxiOut", FloatType()),
    "WHEELS_OFF": ("WheelsOff", FloatType()),
    "WHEELS_ON": ("WheelsOn", FloatType()),
    "TAXI_IN": ("TaxiIn", FloatType()),
    "CRS_ARR_TIME": ("CRSArrTime", IntegerType()),
    "ARR_TIME": ("ArrTime", FloatType()),
    "ARR_DELAY": ("ArrDelay", FloatType()),
    "CANCELLED": ("Cancelled", FloatType()),
    "CANCELLATION_CODE": ("CancellationCode", StringType()),
    "DIVERTED": ("Diverted", FloatType()),
    "CRS_ELAPSED_TIME": ("CRSElapsedTime", FloatType()),
    "ACTUAL_ELAPSED_TIME": ("ActualElapsedTime", FloatType()),
    "AIR_TIME": ("AirTime", FloatType()),
    "DISTANCE": ("Distance", FloatType()),
    "CARRIER_DELAY": ("CarrierDelay", FloatType()),
    "WEATHER_DELAY": ("WeatherDelay", FloatType()),
    "NAS_DELAY": ("NASDelay", FloatType()),
    "SECURITY_DELAY": ("SecurityDelay", FloatType()),
    "LATE_AIRCRAFT_DELAY": ("LateAircraftDelay", FloatType())
}

selected_cols = [
    col(src).cast(dtype).alias(dst) for dst, (src, dtype) in column_mapping.items()
]

streaming_df = streaming_df.select(*selected_cols)

#Fill nulls in delay columns
delay_cols = [
    "ARR_DELAY", "DEP_DELAY", "CARRIER_DELAY", "WEATHER_DELAY",
    "NAS_DELAY", "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY"
]
streaming_df = streaming_df.na.fill(0, subset=delay_cols)

#Add Year and Month columns
streaming_df = streaming_df.withColumn("Year", year(col("FL_DATE"))) \
                           .withColumn("Month", month(col("FL_DATE")))

#Add binary delayed flag
streaming_df = streaming_df.withColumn("IsDelayed",when(col("ARR_DELAY") > 15, 1).otherwise(0))

#Drop cancelled flights
streaming_df = streaming_df.filter(col("CANCELLED") == 0).drop("CANCELLATION_CODE")

In [6]:
# Defining the cleaning to process the streaming data
#Output is in "output" folder and the new data is added every 10 sec
cleaning = (
    streaming_df.writeStream
        .trigger(processingTime="10 seconds")
        .option("maxFilesPerTrigger", NUM_FILES_PER_TRIGGER)
        .option("path", "output/flights")
        .option("checkpointLocation", "checkpoint/flights")
        .format("parquet")
        .outputMode("append")
        .partitionBy("Year", "Month")
        .start()
)

25/12/11 23:00:16 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [7]:
#df_try = spark.read.parquet("output/flights/Year=*/Month=*")
#df_try.select("ORIGIN", "DEST", "ARR_DELAY").filter("ARR_DELAY > 60").show(5)

In [8]:
#df_try.groupBy("ORIGIN", "DEST").count().show()

In [10]:
#!!!!!!!!!!!!!!!!!!!!Run this once you are finished, not immidiately!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
#cleaning.stop()